# Setup

In [ ]:
#You might need to install these packages before running. Comment out after
!pip install torch
!pip install transformers datasets
!pip install scikit-learn
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 46.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.2/521.2 kB 50.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.7/311.7 kB 26.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 56.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 21.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.6/248.6 kB 32.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 9.7 MB/s eta 0:00:00


In [ ]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [ ]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/nlpproject/"

Mounted at /content/drive
/content/drive/My Drive/nlpproject


In [ ]:
import wandb
import csv
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM,AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def multi_label_formatting(data):
    """This function does not really play a huge role for us right now, but what it basically does is that if a paragraph
    has two stances, i.e. 'conservative' and 'right', it will keep both for the classification problem.   """
    labels = []
    for i in range(len(data['stance'])):
        multi_tags = []
        if type(data['stance'][i]) is not float:
            multi_tags.append(data['stance'][i].lower())

        labels.append(multi_tags)

    return labels

#Preprocessing the CSV file that contains the BASIL database.
# df = pd.read_csv('processed_data.csv')
df = pd.read_csv('processed_data_combined.csv')

paragraphs = df["body"] # Gets paragraphs from CSV

#This line was intended to convert stances to integers. Is not needed anymore since now we use MultiLabel Binarizer for
# multilabel classification
# df['stance'] = df['stance'].replace([ 'left', 'center', 'liberal', 'conservative', 'right'],[0,1,2,3,4])


stances = df['stance'] # Gets all the stances/labels
# Stances: {'left', 'center', 'liberal', 'conservative', 'right'}
df['body'] =df['body'].astype(str)

In [ ]:
df.head()

,title,body,stance
0,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
1,Mary Cheney: Sister Is 'Dead Wrong' On Gay Mar...,"Mary Cheney, the younger sister of Wyoming U.S...",center
2,IRS official who refused to testify facing mor...,The IRS official who refused to testify at a H...,center
3,White House Plays Down Data Program,WASHINGTON — The Obama administration tried Sa...,center
4,N.R.A. Details Plan for Armed School Guards,Report Sees Guns as Path to Safety in Schools\...,center


In [ ]:
print('dataset size:', df.shape[0])

dataset size: 37854


In [ ]:
# subset = df.sample(n=dataset_size, random_state=0).reset_index(drop=True)
class_size = 200
subset = df.groupby('stance', group_keys=False).apply(lambda x: x.sample(min(len(x), class_size))).reset_index(drop=True)
print(subset.shape)

(600, 3)


In [ ]:
subset.head()

,title,body,stance
0,Facebook CEO Mark Zuckerberg defends decision ...,Facebook CEO and co-founder Mark Zuckerberg on...,center
1,Supreme Court grants NY prosecutors access to ...,The Supreme Court in a split decision on Thurs...,center
2,Why I Am Disappointed With The 2016 Presidenti...,"Nobody ’ s perfect , or so Hannah Montana says...",center
3,Citing 'security concerns' due to government s...,WASHINGTON – House Speaker Nancy Pelosi asked ...,center
4,China Announces Tariff Retaliation to Take Eff...,LISTEN TO ARTICLE 1:50 SHARE THIS ARTICLE Shar...,center


In [ ]:
subset.groupby('stance', group_keys=False).count()

,title,body
stance,,
center,200,200
left,200,200
right,200,200


In [ ]:
def init_data_model(batch_size, class_size, test_size):

    # Use if you would want to print a sample paragraph and label
    # print(df['body'][100])
    # print(df['stance'][100])

    # subset = df.sample(n=dataset_size, random_state=0).reset_index(drop=True)

    # stratefied sampling of the dataset with even number of each class
    subset = df.groupby('stance', group_keys=False).apply(lambda x: x.sample(min(len(x), class_size))).reset_index(drop=True)

    print('Dataset size:', subset.shape[0])
    print(subset.groupby('stance', group_keys=False).count())


    #This package will convert tags to an array of size 5 (five because we have 5 stances:
    # 'left', 'center', 'liberal', 'conservative', 'right') where, for example, if a paragraph is classified as 'center'
    # it converts its label into one hot encoding [0,0,1,0,0]
    mlb = MultiLabelBinarizer()
    labels = multi_label_formatting(subset) # In case of multitags. Look at function description for more info
    print(f"Labels : {labels}")
    #One Hot Enconding of Multi labels
    labels = mlb.fit_transform(labels)

    #Splitting data into test set and training set.
    x_train_og, x_test_og, y_train, y_test = train_test_split(subset['body'].astype(str), labels, test_size=test_size, random_state = 0)

    #These following two models are way bigger and perform worse (tested.)
    # model_name = "roberta-large"
    # model_name = "roberta-base"

    model_name = "launch/POLITICS" # POLITICS model from HuggingFace!

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # You can check that maximum amount of tokes is 512 which means that we will not be able
    # to process the entire paragraphs.
    # print(tokenizer.model_max_length)

    # model = AutoModelForMaskedLM.from_pretrained("launch/POLITICS")
    n_labels = 3 # num_labels = 5 enables hugging face to add a classification head to the model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n_labels)

    train_encodings = tokenizer(x_train_og.to_list(), truncation=True, padding=True, return_tensors="pt")
    train_labels = torch.tensor(y_train, dtype=torch.float32)
    train_dataset = TensorDataset(train_encodings.input_ids, train_encodings.attention_mask, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Dataloader for test data
    test_encodings = tokenizer(x_test_og.to_list(), truncation=True, padding=True, return_tensors="pt")
    test_labels = torch.tensor(y_test, dtype=torch.float32)
    test_dataset = TensorDataset(test_encodings.input_ids, test_encodings.attention_mask, test_labels)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)  # No need to shuffle test data

    return train_loader, test_loader, model, tokenizer, mlb.classes_


def evaluate(test_loader, model, tokenizer, classes=None, report=False):
    # Predict on the test data
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Softmax makes more sense for single classifications
            predictions = outputs.logits.softmax(dim=-1).tolist()
            all_preds.extend(predictions)

            # In case you'd want to use Sigmoid
            # predictions = torch.sigmoid(logits)  # Apply sigmoid activation for multilabel classification
            # all_preds.extend(predictions.cpu().detach().numpy())

            all_labels.extend(labels.cpu().detach().numpy())

    # Convert the predictions and labels to binary values based on a threshold (e.g., 0.5)
    threshold = 0.5

    all_preds = (torch.tensor(all_preds) > threshold).int().numpy()
    all_labels = np.array(all_labels)

    # Compute the classification report
    accuracy = accuracy_score(all_labels, all_preds)

    # Reporting Results
    if report:
      #Bigger report summary. Sample avg is the same as Accuracy.
      report = classification_report(all_labels, all_preds, target_names=classes)
      print(report)

    # return more things want more information
    return accuracy


def train(train_loader, test_loader, model, tokenizer, lr, batch_size, threshold):
    #Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # # Fine-tuning loop
    model.to(device)

    num_epochs = 20

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids, attention_mask, labels = batch
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            # # Backward pass and optimization
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_acc = evaluate(train_loader, model, tokenizer)
        val_acc = evaluate(test_loader, model, tokenizer)

        print(f"train_acc: {train_acc}")
        print(f"val_acc: {val_acc}")

        wandb.log({
            'loss': total_loss,
            'train_acc': train_acc,
            'val_acc': val_acc,
          })

        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss}")
        # test_model() Use if you would want to take a look at how the model is currently doing on the test data. BAD PRACTICE!
        if total_loss < threshold:
          break

    return model, tokenizer

# One-off training
This section is for if you just want to train a single model with a given configuration. Record your configuration in the following wandb config, and simply run the training block. The loss will be reported to wandb.

In [ ]:
lr = 7.452711267578379e-06
batch_size = 8
test_size = 0.1
threshold = 1.5
class_size = 800

wandb.init(
    project='politics_more_data',
    config= {
        'learning_rate': lr,
        'batch_size': batch_size,
        'test_size': test_size,
        'threshold': threshold,
        'class_size': class_size,
    }
)

wandb: WARNING Ignored wandb.init() arg project when running a sweep.


loss,█▅▄▃▂▁▁▁▁▁
train_acc,▁▅▆▇██████
val_acc,▁▆▇███▇██▆
loss,8.0928
train_acc,0.99398
val_acc,0.80417


In [ ]:
# Load model directly
train_loader, test_loader, model, tokenizer, classes = init_data_model(batch_size, class_size, test_size)

Dataset size: 2400
        title  body
stance             
center    800   800
left      800   800
right     800   800
Labels : [['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['cente

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
model, tokenizer = train(train_loader, test_loader, model, tokenizer, lr, batch_size, threshold)

train_acc: 0.9837962962962963
val_acc: 0.8083333333333333
Epoch 11/20, Loss: 8.418230362702161
train_acc: 0.6375
val_acc: 0.55
Epoch 1/20, Loss: 151.07501140236855
train_acc: 0.9861111111111112
val_acc: 0.7708333333333334
Epoch 12/20, Loss: 6.6410349467769265
train_acc: 0.8319444444444445
val_acc: 0.675
Epoch 2/20, Loss: 100.16493974626064
train_acc: 0.9958333333333333
val_acc: 0.825
Epoch 13/20, Loss: 8.053604286629707
train_acc: 0.9949074074074075
val_acc: 0.8333333333333334
Epoch 14/20, Loss: 4.478643105830997
train_acc: 0.9171296296296296
val_acc: 0.7458333333333333
Epoch 3/20, Loss: 66.13009167462587
train_acc: 0.9972222222222222
val_acc: 0.8625
Epoch 15/20, Loss: 4.368915635626763
train_acc: 0.9300925925925926
val_acc: 0.7458333333333333
Epoch 4/20, Loss: 42.68605802208185


In [ ]:
# final evaluation
acc = evaluate(test_loader, model, tokenizer, classes, report=True)

              precision    recall  f1-score   support

      center       0.92      0.44      0.59        25
        left       0.86      0.74      0.79        34
       right       0.59      0.94      0.72        31

   micro avg       0.72      0.72      0.72        90
   macro avg       0.79      0.70      0.70        90
weighted avg       0.78      0.72      0.71        90
 samples avg       0.72      0.72      0.72        90



In [ ]:
# save the model
save_name = 'politics_best'

model.save_pretrained(save_name)
# tokenizer.save_pretrained(save_name+'_tokenizer')

In [ ]:
wandb.finish()

loss,█▆▃▂▁▂▁▁
train_acc,▁▇▆██▆█▇
val_acc,▁█▆█▇▅▆█
loss,1.40882
train_acc,0.94568
val_acc,0.73333


# Hyperparameter Fine-tuning
This section is for doing sweeps over different hyperparameters to fine-tune for the best accuracy.

In [ ]:
sweep_config = {
    'method': 'random',
    'name': 'sweep',
    'metric': {'goal': 'maximize', 'name': 'val_acc'},
    'parameters': {
        'batch_size': {'values': [8, 16]},
        'lr': {'max': 0.00006, 'min': 1e-7},
        # 'test_size': {'values': [0.2, 0.25, 0.3]},
        'threshold': {'values': [1, 1.5, 2, 2.5, 3, 3.5, 4]},
        'class_size': {'values': list(range(100, 1001, 50))},
    }
}

sweep_id = wandb.sweep(sweep=sweep_config, project='politics-sweep')

Create sweep with ID: kudip81h
Sweep URL: https://wandb.ai/probgram/politics-sweep/sweeps/kudip81h


In [ ]:
def main():
  run = wandb.init()

  lr = wandb.config.lr
  batch_size = wandb.config.batch_size
  # test_size = wandb.config.test_size
  test_size = 0.1
  threshold = wandb.config.threshold
  class_size = wandb.config.class_size

  train_loader, test_loader, model, tokenizer, classes = init_data_model(batch_size, class_size, test_size)
  model, tokenizer = train(train_loader, test_loader, model, tokenizer, lr, batch_size, threshold)

  # del test_labels
  del model
  del tokenizer
  torch.cuda.empty_cache()

  # return model, tokenizer


In [ ]:
wandb.agent(sweep_id, function=main, count=20)

wandb: Agent Starting Run: wv3saty5 with config:
wandb: 	batch_size: 8
wandb: 	class_size: 800
wandb: 	lr: 7.452711267578379e-06
wandb: 	threshold: 1.5
wandb: Currently logged in as: ellieyhc (probgram). Use `wandb login --relogin` to force relogin


Dataset size: 2400
        title  body
stance             
center    800   800
left      800   800
right     800   800
Labels : [['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['center'], ['cente

(…)ITICS/resolve/main/tokenizer_config.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

(…)/launch/POLITICS/resolve/main/vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

(…)/launch/POLITICS/resolve/main/merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

(…)launch/POLITICS/resolve/main/config.json:   0%|          | 0.00/672 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at launch/POLITICS and are newly initialized: ['classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


train_acc: 0.6712962962962963
val_acc: 0.7
Epoch 1/20, Loss: 156.35882025957108
train_acc: 0.8601851851851852
val_acc: 0.8083333333333333
Epoch 2/20, Loss: 98.29894362390041
train_acc: 0.924074074074074
val_acc: 0.8291666666666667
Epoch 3/20, Loss: 62.782889023423195
train_acc: 0.9699074074074074
val_acc: 0.8375
Epoch 4/20, Loss: 40.056127082556486
train_acc: 0.9745370370370371
val_acc: 0.8375
Epoch 5/20, Loss: 24.627702036872506
train_acc: 0.9884259259259259
val_acc: 0.8416666666666667
Epoch 6/20, Loss: 17.85879083070904
train_acc: 0.9898148148148148
val_acc: 0.8125
Epoch 7/20, Loss: 13.804262374527752
train_acc: 0.9837962962962963
val_acc: 0.8375
Epoch 8/20, Loss: 11.874859482049942


wandb: Ctrl + C detected. Stopping sweep.


In [ ]:
wandb.finish()

# Prediction
This is hacky, someone should write a prediction function properly

In [ ]:
model_name = '/content/drive/MyDrive/NLP_Research_Project/politics_best'
batch_size = 16

In [ ]:
val_df = df.sample(n=200).reset_index(drop=True)
val_df.head()

,title,body,stance
0,Bitter exchanges and incriminating evidence ro...,Washington ( CNN ) Late-night rancor erupted a...,left
1,Here's how technology can help reduce politica...,"“ They voted for Trump , so obviously we can ’...",center
2,Let’s Say Russia Did Hack the Dems. What Would...,From the perspective of a Gary Johnson voter w...,right
3,Trump lawyer dismisses tax return demand,Donald Trump has the right to keep his tax ret...,center
4,Revised Senate health care bill already facing...,"In October 2016 , Donald Trump told the enthus...",right


In [ ]:
mlb = MultiLabelBinarizer()
labels = multi_label_formatting(val_df) # In case of multitags. Look at function description for more info
print(f"Labels : {labels}")
#One Hot Enconding of Multi labels
labels = mlb.fit_transform(labels)

X, _, y, _ = train_test_split(val_df['body'].astype(str), labels,test_size=1)

tokenizer = AutoTokenizer.from_pretrained("launch/POLITICS")

n_labels = 3 # num_labels = 5 enables hugging face to add a classification head to the model
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)

X_enc = tokenizer(X.to_list(), truncation=True, padding=True, return_tensors="pt")
y_labels = torch.tensor(y, dtype=torch.float32)
X_dataset = TensorDataset(X_enc.input_ids, X_enc.attention_mask, y_labels)
X_loader = DataLoader(X_dataset, batch_size=batch_size)

Labels : [['left'], ['center'], ['right'], ['center'], ['right'], ['right'], ['left'], ['center'], ['center'], ['center'], ['left'], ['right'], ['left'], ['right'], ['left'], ['center'], ['left'], ['left'], ['left'], ['center'], ['right'], ['left'], ['center'], ['center'], ['center'], ['right'], ['right'], ['right'], ['right'], ['right'], ['right'], ['right'], ['right'], ['left'], ['left'], ['right'], ['left'], ['center'], ['center'], ['right'], ['left'], ['left'], ['left'], ['center'], ['center'], ['right'], ['right'], ['center'], ['center'], ['left'], ['left'], ['center'], ['right'], ['center'], ['right'], ['right'], ['left'], ['center'], ['center'], ['right'], ['center'], ['left'], ['right'], ['right'], ['right'], ['right'], ['center'], ['right'], ['left'], ['left'], ['left'], ['center'], ['right'], ['center'], ['right'], ['left'], ['right'], ['left'], ['left'], ['right'], ['center'], ['left'], ['left'], ['right'], ['center'], ['left'], ['right'], ['right'], ['right'], ['center'], [

In [ ]:
acc = evaluate(X_loader, model, tokenizer, mlb.classes_, report=True)
print(acc)

              precision    recall  f1-score   support

      center       0.88      0.50      0.64        60
        left       0.78      0.75      0.76        67
       right       0.65      0.92      0.76        72

   micro avg       0.73      0.73      0.73       199
   macro avg       0.77      0.72      0.72       199
weighted avg       0.77      0.73      0.73       199
 samples avg       0.73      0.73      0.73       199

0.7336683417085427


In [ ]:
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in X_loader:
        input_ids, attention_mask, labels = batch
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Softmax makes more sense for single classifications
        predictions = outputs.logits.softmax(dim=-1).tolist()
        all_preds.extend(predictions)

        # In case you'd want to use Sigmoid
        # predictions = torch.sigmoid(logits)  # Apply sigmoid activation for multilabel classification
        # all_preds.extend(predictions.cpu().detach().numpy())

        all_labels.extend(labels.cpu().detach().numpy())

# Convert the predictions and labels to binary values based on a threshold (e.g., 0.5)
threshold = 0.5

all_preds = (torch.tensor(all_preds) > threshold).int().numpy()
all_labels = np.array(all_labels)

In [ ]:
all_X_y = pd.DataFrame(np.column_stack((X, mlb.inverse_transform(all_labels), mlb.inverse_transform(all_preds))))

In [ ]:
correct_preds = all_X_y[all_X_y[1] == all_X_y[2]]
print(correct_preds.shape[0])
correct_preds.head()

146


,0,1,2
2,Theresa May has bowed to intense pressure from...,left,left
3,As President Obama launches into the next phas...,right,right
5,Story highlights Senate Majority Leader Reid s...,left,left
6,Sen. John McCain said Thursday he is worried a...,right,right
8,Remember Merrick Garland ? He ’ s the appeals ...,left,left


In [ ]:
incorrect_preds = all_X_y[all_X_y[1] != all_X_y[2]]
print(incorrect_preds.shape[0])
incorrect_preds.head()

53


,0,1,2
0,"Previewing a rancorous fall campaign , Hillary...",center,right
1,In the weeks leading up to South Sudan ’ s ind...,left,right
4,WASHINGTON ( AP ) — Acting Attorney General Ma...,center,right
7,LAS VEGAS— Donald Trump refused Wednesday to c...,center,left
15,The state funeral for former President George ...,center,left


# Free up memory

In [ ]:
#Memory Management
# del df
# del test_labels
del model
del tokenizer
torch.cuda.empty_cache()